In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import brentq

from IPython.display import Image


**Kolmogorov's energy spectrum in the inertial subrange** (a steady-state regime in energy spectrum)

${
E(k) = E_0 \, 
            k^{-5/3}
}
$
where, 

${
E_0 = 1.5 \, 
            \epsilon^{2/3} 
            \left[ 
                1 - \left(\frac{\eta}{L}\right) ^{4/3}
            \right]^{-1}
}
$
with kinematic viscocity 
$
\eta = {\left( \frac{\nu^3}{\epsilon} \right)}^{1/4}
$

**Set all parameters for the simulations**

In [ ]:
### Parameters for velocity field (in SI units * 1e6 to convert to mm)

# Dissipation rate 
epsilon = 1e-4 * 1e6 # Ranges from 1e-4 (rough sea) to 1e-14 (calm sea) - *1e-6 to convert to mm

# Kinematic viscosity of sea water depends on salinity and temperature
nu = 1e-6 * 1e6 # Ranges from 1e-6 to 1.8e-6, times 1e-6 to convert to mm

# Maximum length scale of turbulence
L = 90 # [mm]

# Total number of wave numbers sampled (capped by resolution of simulation)
N = 50

### Simulation parameters
tank_size = 90
grid_size = tank_size * 2 + 1  # 0.1 mm grid spacing
fps = 20 # temporal resolution [fps]
t_simulation = 60 # total length of simulation [s]
N_plankton = 100 # Number of plankters

### Video parameters
stride = 6
step = 1

### Create grid for the simulation
x = np.linspace(0, tank_size, grid_size)
y = np.linspace(0, tank_size, grid_size)
X, Y = np.meshgrid(x, y)

### Calculate timesteps for the animation
timesteps = t_simulation * fps

### Set up the meshgrid for plotting the velocity field
Xs = X[::stride, ::stride]
Ys = Y[::stride, ::stride]

### 1. Setting up the Kolmogorov energy speectrum


In [ ]:
# Define eta (Kolmogorov length scale)
eta = (nu**3 / epsilon) ** (1/4)   

# Calculate E0 (amplitude of energy spectrum)
E0 = 1.5 * epsilon**(2/3) * (1 - (eta/L)**(4/3))**(-1) 

# Linear sampling of wavenumbers fir visualization
k_min = 2 * np.pi / L
k_max = 2 * np.pi / eta
k = np.linspace(k_min, k_max, 500)

# Calculate energy spectrum
E = E0 * (k**(-5/3))

# # Plot
plt.figure(figsize=(10, 6))
plt.loglog(k, E, label=r'$E(k) = E_0 \cdot k^{-5/3}$')
plt.xlabel('Wave number $k$')
plt.ylabel('Energy spectrum $E(k)$')
plt.title("Kolmogorov Energy Spectrum in the Inertial Range")
plt.legend()
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.show()

**Sampling wave numbers distributed in a geometric series from $k_1$ to $k_N$, N is maximum amount of waves that the grid can show**

In [ ]:
# Generate wavevectors k_n using the geometric series in ascending order
dx = tank_size / (grid_size - 1)          # 1 mm grid spacing
k_min = 2 * np.pi / L
k_max_grid = np.pi / dx                    # Nyquist limit of the grid, ~3.14 rad/mm
k_max_resolved = min(2 * np.pi / eta, k_max_grid)   # don't sample modes finer than the grid can show

k_values = k_min * (k_max_resolved / k_min) ** (np.arange(N) / (N - 1))

# Energy spectrum
E_k = E0 * (k_values ** (-5 / 3))

# Delta k values
delta_k0 = (k_values[1] - k_values[0]) / 2
delta_kN = (k_values[-1] - k_values[-2]) / 2
delta_k_ = (k_values[2:] - k_values[:-2]) / 2
delta_k = np.concatenate(([delta_k0], delta_k_, [delta_kN]))

# Plot the Kolmogorov energy specutrm in the intertial subrange
plt.figure(figsize=(10, 6))
plt.loglog(k_values, E_k, marker='o', label=r'$E(k) = E_0 \cdot k^{-5/3}$', color="darkviolet")
plt.xlabel('Wave number $k$')
plt.ylabel('Energy spectrum $E(k)$')
plt.title("Kolmogorov Energy Spectrum in the Inertial Range")
plt.legend()
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.show()

### 2. **Kinematic simulation of velocity fields**

Simulate the velocity field based on kolmogorov energy spectrum. The magnitude of a field at position x at time t is found as below

$$\mathbf{u}(\mathbf{x}, t) = \sum_{n=1}^{N} \left[ \mathbf{A}_n \cos(\mathbf{k}_n \cdot \mathbf{x} + \omega_n t) + \mathbf{B}_n \sin(\mathbf{k}_n \cdot \mathbf{x} + \omega_n t) \right]$$

where


- $\mathbf{u}(\mathbf{x}, t)$ : the simulated 2D velocity field, evaluated at position $\mathbf{x}$ and time $t$ (`velocity_field` in the code).
- $\mathbf{x} = (x, y)$ : the spatial position at which the field is evaluated (the grid points defined by `X`, `Y`).
- $t$ : time (the timestep index in the loop).
- $N$ : the total number of Fourier (wave) modes summed to build the field, sampled geometrically between the largest scale $k_{min} = 2\pi/L$ and the Kolmogorov scale $k_{max} = 2\pi/\eta$.
- $n$ : the index of an individual Fourier mode, $n = 1, \dots, N$.
- $\mathbf{k}_n$ : the wavevector of mode $n$.
- $\omega_n$ : the angular (temporal) frequency of mode $n$
- $\mathbf{A}_n, \mathbf{B}_n$ : the vector amplitude coefficients of mode $n$ 

In [ ]:
# Amplitudes of the Fourier modes
a_n = b_n = np.sqrt(2 * E_k * delta_k)

# Temporal frequencies
omega_n = 0.4 * np.sqrt((k_values ** 3) * E_k)

# Random phases for amplitudes and wavevectors
angles = 2 * np.pi * np.random.rand(N)

# Define A_n, B_n, and k_n according to the incompressibility constraints (Shape: (N, 2)), where N = number of Fourier modes
A_n = np.array([a_n * np.cos(angles), -a_n * np.sin(angles)]).T
B_n = np.array([-b_n * np.cos(angles), b_n * np.sin(angles)]).T
k_n = np.array([k_values * np.sin(angles), k_values * np.cos(angles)]).T

# Define the spatial grid (x, y)
positions = np.stack([X.ravel(), Y.ravel()], axis=-1)  # Flattened grid positions for efficiency (Shape: (grid_size * grid_size, 2))

# Initialize an array to store the velocity field at each time step
velocity_field = np.zeros((timesteps, grid_size, grid_size, 2))

# Compute the velocity field for each time step
for t in range(timesteps):
    print(f"\r Simulating velocity field for timestep {t}/{timesteps} ...", end='\r', flush=True)
    # Initialize the velocity at each point to zero
    u = np.zeros((grid_size * grid_size, 2))

    # Loop over each Fourier mode
    for n in range(N):
        # Compute the phase shift for each mode
        phase = np.dot(positions, k_n[n]) + omega_n[n] * (t/fps)

        # Reshape A_n[n] and B_n[n] to be (1, 2) for broadcasting (for multiplication to all points in the grid)
        A_n_n = A_n[n].reshape(1, 2)
        B_n_n = B_n[n].reshape(1, 2)

        # Add the contribution of this mode to the velocity field
        u += (A_n_n * np.cos(phase)[:, None] + B_n_n * np.sin(phase)[:, None])


    # Reshape and store the velocity field for this time step
    velocity_field[t, ..., 0] = u[:, 0].reshape(grid_size, grid_size)  # x-component
    velocity_field[t, ..., 1] = u[:, 1].reshape(grid_size, grid_size)  # y-component

print("finished")
# velocity_field now contains the velocity vectors at each point on the grid for each time step

Plot quiver plot of velocity field

In [ ]:
# Select a time step
t = 0  # Choose a specific time step (e.g., t=0 for the initial field)

Us = velocity_field[t, ::stride, ::stride, 0]  # x-component of the velocity
Vs = velocity_field[t, ::stride, ::stride, 1]  # y-component of the velocity

# Plot the velocity field using quiver
plt.figure(figsize=(12, 12))
plt.quiver(Xs, Ys, Us, Vs, scale=1000, pivot='mid', color='blue')
plt.title(f"Velocity Field at Time Step {t}")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.grid()
plt.show()

U = velocity_field[t, ..., 0]  # x-component of the velocity
V = velocity_field[t, ..., 1]  # y-component of the velocity
plt.figure(figsize=(8, 8))
plt.streamplot(X, Y, U, V, density=1.5, color=np.sqrt(U**2 + V**2), cmap='viridis')
plt.colorbar(label="Velocity Magnitude")
plt.title(f"Streamline Plot of Velocity Field at Time Step {t}")
plt.xlabel("x")
plt.ylabel("y")
plt.grid()
plt.show()

Quiver plot of velocity fields: Video

In [ ]:
# Set up the figure and axis for the quiver plot
fig, ax = plt.subplots(figsize=(8, 8))

frame_indices = range(0, timesteps, step)

### Plot quiver plot simulation
U0, V0 = velocity_field[0, ::stride, ::stride, 0], velocity_field[0, ::stride, ::stride, 1]

fig, ax = plt.subplots(figsize=(8, 8))
quiver = ax.quiver(Xs, Ys, U0, V0, scale=1000, pivot='mid', color='blue')
title = ax.set_title("Velocity Field at T = 0 s")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid()

def update_quiver(frame):
    print(f"\rProcessing frame {frame}/{timesteps} ...", end='', flush=True)
    U = velocity_field[frame, ::stride, ::stride, 0]
    V = velocity_field[frame, ::stride, ::stride, 1]
    quiver.set_UVC(U, V)
    title.set_text(f"Velocity Field at T = {frame/fps}.2f s")
    return quiver, title

anim = FuncAnimation(fig, update_quiver, frames=frame_indices, interval=50, blit=True)

gif_path = "velocityfield_quiver.gif"
anim.save(gif_path, writer=PillowWriter(fps=fps/step))
plt.close(fig)
Image(filename=gif_path)


Verifying the interpolation

**Note**: Scipy grid interpolator expects "physical" coordinates as inputs, not indices.

In [ ]:
# Select a timestep for testing
timestep = 0
u_interpolator = RegularGridInterpolator((x, y), velocity_field[timestep, ..., 0])  # x-component of velocity
v_interpolator = RegularGridInterpolator((x, y), velocity_field[timestep, ..., 1])  # y-component of velocity

# Check interpolation at a specific grid point by using coordinates, not indices
index = 3  # Example grid index (4th row and 4th column in zero-based index)
point = [x[index], y[index]]  # Use actual physical coordinates

# Interpolated velocity at this physical point
u_point = u_interpolator(point)
v_point = v_interpolator(point)
print(f"Interpolated velocity at point {point}: ({u_point}, {v_point})")

# Compare with the actual velocity at the closest grid point using indices
u_grid = velocity_field[timestep, index, index, 0]
v_grid = velocity_field[timestep, index, index, 1]
print(f"Velocity at grid index [{index}, {index}]: ({u_grid}, {v_grid})")


### 3. Coupling Planktons to turbulence field (Stocastic differential equations)

**Extract starting parameters from the measurements**

In [ ]:
### Read the data from experiments
measurement = "Artemia_0805"
path_to_still = r"R:\\LU24A1047-PLS\\TrackingData\\" + measurement + r"\\trajectories_scaled\\traj_still_scaled.pkl"
path_to_control = r"R:\\LU24A1047-PLS\\TrackingData\\" + measurement + r"\\trajectories_scaled\\traj_nothing_scaled.pkl"

df_still = pd.read_pickle(path_to_still)
df_control = pd.read_pickle(path_to_control)


df_still = df_still.dropna(subset=["speed", "orientation"])
df_control = df_control.dropna(subset=["speed", "orientation"])

# velocity: mean directed swimming speed from the stimulus condition [mm/s]
velocity_measurement = df_still["speed"].mean()

# response_angle: half-width of a uniform heading spread around target_angle (90 deg)
# that reproduces the measured resultant length R of the still condition
target_angle = np.pi / 2
R_still = np.sqrt(np.mean(np.cos(df_still["orientation"])) ** 2 + np.mean(np.sin(df_still["orientation"])) ** 2)
response_angle_measurement = brentq(lambda a: np.sinc(a / np.pi) - R_still, 1e-6, np.pi - 1e-6)

# Dt: random swimming component from the no-stimulus control condition [mm^2/s]
# matches mean-squared displacement over one frame interval (delta_t) to the model's 2*Dt*delta_t
delta_t_meas = 0.1  # video frame interval [s]
Dt_measurement = np.mean(df_control["speed"] ** 2) * delta_t_meas / 4

print(f"velocity = {velocity_measurement:.4f} mm/s")
print(f"response_angle = {np.degrees(response_angle_measurement):.2f} deg")
print(f"Dt = {Dt_measurement:.4e} mm^2/s")


**Set plankton parameters**

In [ ]:
### Set Plankton parameters
set_turbulence = True

### Starting conditions of plankton (based on real swimmers)
velocity = velocity_measurement # mean upward velocity [mm/s]
Dt = Dt_measurement
angle = response_angle_measurement # in degrees (plus or minus around the target angle 90 degrees)

**Coupling the velocity field with the plankton swimming and random noise**

In [ ]:
### Frame rate (10 by default)
delta_t = 1/fps

target_angle = np.pi / 2
response_angle = angle * (np.pi / 180)

# Define velocity interpolators
x_grid = np.linspace(0, tank_size, grid_size)
y_grid = np.linspace(0, tank_size, grid_size)

def get_velocity_interpolators(time_step):
    vx_interpolator = RegularGridInterpolator((x_grid, y_grid), velocity_field[time_step, ..., 0])
    vy_interpolator = RegularGridInterpolator((x_grid, y_grid), velocity_field[time_step, ..., 1])
    return vx_interpolator, vy_interpolator

# Store the positions
stored_positions = np.zeros((n_planktons, 2, timesteps))

# Intialize the plankton positions and orientations
# x_pos = np.random.rand(n_planktons) * L
# y_pos = np.random.rand(n_planktons) * L
x_pos = np.random.uniform(0, tank_size, n_planktons)
y_pos = np.random.uniform(0, tank_size, n_planktons)
phi = np.random.rand(n_planktons) * 2 * np.pi

for t in range(timesteps):
    print(f"\r Simulating plankton movement for timestep {t}/{timesteps} ...", end='', flush=True)
    phi = target_angle + response_angle * (2 * np.random.rand(n_planktons) - 1)

    # turbulence velocities
    vx_interpolator, vy_interpolator = get_velocity_interpolators(t)
    positions = np.stack([x_pos, y_pos], axis=-1)
    vx_turb = vx_interpolator(positions)
    vy_turb = vy_interpolator(positions)

    if set_turbulence is False:
        vx_turb = 0
        vy_turb = 0

    # behavioral velocities
    vx_behav = velocity * np.cos(phi)
    vy_behav = velocity * np.sin(phi)

    # update positions
    x_pos = x_pos + (vx_turb + vx_behav) * delta_t + np.sqrt(2 * Dt * delta_t) * np.random.randn(n_planktons)
    y_pos = y_pos + (vy_turb + vy_behav) * delta_t + np.sqrt(2 * Dt * delta_t) * np.random.randn(n_planktons)

    # reflect at the boundaries
    x_pos[x_pos > tank_size] = 2 * tank_size - x_pos[x_pos > tank_size]
    x_pos[x_pos < 0] = -x_pos[x_pos < 0]
    y_pos[y_pos > tank_size] = 2 * tank_size - y_pos[y_pos > tank_size]
    y_pos[y_pos < 0] = -y_pos[y_pos < 0]

    # store the positions
    stored_positions[:, 0, t] = x_pos
    stored_positions[:, 1, t] = y_pos

**Create animation of velocity field with plankton swimming through it**

In [ ]:
# Video parameters
frame_indices = range(0, timesteps, step)
trail_length = 20  # Number of previous positions to show in the trail

# Set up the figure and axis for the quiver plot
fig, ax = plt.subplots(figsize=(8, 8))

Xs = X[::stride, ::stride]
Ys = Y[::stride, ::stride]

U = velocity_field[0, ::stride, ::stride, 0]
V = velocity_field[0, ::stride, ::stride, 1]

# Initialize the quiver plot with the first time step
quiver = ax.quiver(Xs, Ys, U, V, scale=1000, pivot='mid', color='blue', alpha=0.2, label='Velocity Field')
title = ax.set_title("T = 0 s")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")

# Add the plankton positions
scat = ax.scatter(stored_positions[:, 0, 0], stored_positions[:, 1, 0], c='black', s=30, alpha=0.5, label='Plankton')
trails = [ax.plot([], [], '-', linewidth=1, color="black", alpha=0.5)[0] for _ in range(n_planktons)]
ax.set_xlim(0, tank_size)
ax.set_ylim(0, tank_size)
ax.legend(loc='upper right')  # created once, not touched again

# Function to update the plot for each frame
def update(frame):
    print(f"\rProcessing frame {frame}/{timesteps} ...", end='', flush=True)
    U = velocity_field[frame, ::stride, ::stride, 0]
    V = velocity_field[frame, ::stride, ::stride, 1]
    quiver.set_UVC(U, V)
    title.set_text(f"T = {frame/fps:.2f} s")
    scat.set_offsets(np.c_[stored_positions[:, 0, frame], stored_positions[:, 1, frame]])

    start = max(0, frame - trail_length)
    for i in range(n_planktons):
        trails[i].set_data(stored_positions[i, 0, start:frame+1], stored_positions[i, 1, start:frame+1])
    return (quiver, scat, title, *trails)

anim = FuncAnimation(fig, update, frames=frame_indices, interval=50, blit=True)

gif_path = "velocityfield_plankton_quiver.gif"
anim.save(gif_path, writer=PillowWriter(fps=fps/step))
plt.close(fig)
Image(filename=gif_path)